# Image Recommendation System

This notebook documents and demonstrates the visual-similarity recommendation project. The system uses a pretrained CNN (ResNet) to extract image embeddings and measure similarity via cosine.

Pipeline: image collection → ResNet embedding extraction → L2 normalization → cosine similarity → top-K results.

Main components:
- `ImageEmbedder`: extracts the image embeddings.
- `index_images`: walks a directory, generates embeddings and saves the index (`index.npz`).
- `recommend`: queries the index and returns the most similar items.
- `write_jsonl`: persists the output as JSONL.

See the README for the full CLI and testing instructions.

## Installation

Install the dependencies (if needed):

```
pip install torch torchvision pillow numpy python-dotenv
```

Optional (Jupytext):

```
jupytext --version
```

### Library Imports

In [91]:
# Standard library imports
import os
import sys
import argparse
import json
import time
import tempfile
import shutil
from typing import List, Tuple

# Third-party imports for array and image handling
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import subprocess

In [92]:
try:
    # Tries to load environment variables from a .env file, if it exists
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    # Ignores it if the dotenv library is not installed or fails
    pass

### Deep Learning Imports

In [93]:
# PyTorch and Torchvision imports for computer vision models
import torch
import torchvision.transforms as T
from torchvision import models

### Timing Utility

In [94]:
class Timer:
    """Utility class that measures execution time."""
    def __init__(self):
        # Records the start time on instantiation
        self.t = time.time()
    def tick(self) -> float:
        # Computes the time difference since the last call or initialization
        now = time.time()
        d = now - self.t
        self.t = now
        return d

### Global Settings

In [95]:
# Defines the image extensions supported by the system
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

### Image Listing

In [96]:
def list_images(root: str) -> List[str]:
    """
    Scans a directory recursively for images with supported extensions.
    """
    if not os.path.isdir(root):
        raise FileNotFoundError(f"Directory not found: {root}")
    out = []
    for base, _, files in os.walk(root):
        for f in files:
            ext = os.path.splitext(f)[1].lower()
            if ext in SUPPORTED_EXTS:
                out.append(os.path.join(base, f))
    if not out:
        raise RuntimeError("No supported image was found")
    # Returns the sorted list to guarantee consistency in the index
    return sorted(out)

### Image Preprocessing

In [97]:
def _default_transforms() -> T.Compose:
    """
    Defines the default transformations that feed the ResNet.
    """
    return T.Compose([
        T.Resize(256),       # Resizes the image
        T.CenterCrop(224),   # Crops the 224x224 center
        T.ToTensor(),        # Converts to a PyTorch Tensor
        # Normalizes with the ImageNet means and standard deviations
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

### Embedding Extraction Engine

In [98]:
class ImageEmbedder:
    """
    Class responsible for loading the model and extracting feature vectors (embeddings).
    """
    def __init__(self, device: str = "cpu", model_name: str = "resnet50"):
        # Defines the device (CPU or GPU)
        self.device = torch.device(device if torch.cuda.is_available() or device == "cpu" else "cpu")
        self.model_name = model_name
        # Loads the model and the transformations
        self.model, self.transform = self._load_model_and_transform()
        self.model.eval() # Evaluation mode (does not train)
        self.model.to(self.device) # Moves to the right device

    def _load_model_and_transform(self):
        """Loads the pretrained ResNet architecture."""
        weights = None
        transform = None
        model = None
        try:
            # Tries to load with the updated ImageNet weights
            if self.model_name == "resnet50":
                weights = models.ResNet50_Weights.IMAGENET1K_V2
                model = models.resnet50(weights=weights)
                transform = weights.transforms()
            elif self.model_name == "resnet18":
                weights = models.ResNet18_Weights.IMAGENET1K_V1
                model = models.resnet18(weights=weights)
                transform = weights.transforms()
            else:
                # Fallback to ResNet50
                weights = models.ResNet50_Weights.IMAGENET1K_V2
                model = models.resnet50(weights=weights)
                transform = weights.transforms()
        except Exception:
            # Fallback to None weights and manual transforms if it fails (e.g. no internet or an old version)
            if self.model_name == "resnet18":
                model = models.resnet18(weights=None)
            else:
                model = models.resnet50(weights=None)
            transform = _default_transforms()
        
        # Removes the last layer (classifier) to keep only the features
        backbone = torch.nn.Sequential(*list(model.children())[:-1])
        return backbone, transform

    def embed(self, image_path: str) -> np.ndarray:
        """Generates the embedding vector for a single image."""
        if not os.path.isfile(image_path):
            raise FileNotFoundError(f"File not found: {image_path}")
        try:
            img = Image.open(image_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed to open image: {image_path}: {str(e)}")
            
        # Runs inference without computing gradients
        with torch.inference_mode():
            x = self.transform(img).unsqueeze(0).to(self.device) # Adds the batch dimension
            feat = self.model(x)
            feat = feat.view(feat.size(0), -1) # Flatten
            v = feat[0].detach().cpu().numpy().astype(np.float32) # Converts to numpy
        return v

### Data Persistence

In [99]:
def save_index(embeddings: np.ndarray, paths: List[str], out_path: str):
    """Saves the embeddings and the file paths in a compressed .npz file."""
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    # Saves compressed numpy arrays
    np.savez_compressed(out_path, embeddings=embeddings, paths=np.array(paths))

### Data Loading

In [100]:
def load_index(index_path: str) -> Tuple[np.ndarray, List[str]]:
    """Loads the embedding index from disk."""
    if not os.path.isfile(index_path):
        raise FileNotFoundError(f"Index not found: {index_path}")
    z = np.load(index_path, allow_pickle=True)
    emb = z["embeddings"].astype(np.float32)
    paths = list(z["paths"].tolist())
    # Basic validation of data integrity
    if emb.ndim != 2 or len(paths) != emb.shape[0]:
        raise RuntimeError("Invalid index")
    return emb, paths

### Vector Normalization

In [101]:
def normalize_rows(mat: np.ndarray) -> np.ndarray:
    """Normalizes the rows of a matrix (L2 norm) for the cosine similarity computation."""
    # Computes the L2 norm of each row
    n = np.linalg.norm(mat, axis=1, keepdims=True)
    # Avoids division by zero
    n[n == 0] = 1.0
    return mat / n

### Main Indexing Function

In [102]:
def index_images(image_dir: str, out_path: str, device: str = "cpu", model_name: str = "resnet50", batch_size: int = 1) -> Tuple[np.ndarray, List[str]]:
    """Full indexing process: lists images -> generates embeddings -> saves."""
    timer = Timer()
    paths = list_images(image_dir)
    embeder = ImageEmbedder(device=device, model_name=model_name)
    vecs = []
    
    # Iterates over all images found
    for p in paths:
        try:
            v = embeder.embed(p)
            vecs.append(v)
        except Exception as e:
            # Logs the error but continues the process
            print(json.dumps({"erro": str(e), "arquivo": p}))
    
    if not vecs:
        raise RuntimeError("No embedding was generated")
    
    # Stacks the vectors into a matrix
    E = np.vstack(vecs)
    # Normalizes to enable dot-product search (cosine)
    E = normalize_rows(E)
    
    # Saves the result
    save_index(E, paths[:E.shape[0]], out_path)
    
    # Prints statistics
    print(json.dumps({"tempo_segundos": round(timer.tick(), 3), "imagens_indexadas": E.shape[0], "dim": int(E.shape[1])}))
    return E, paths[:E.shape[0]]

### Main Recommendation Function

In [103]:
def recommend(index_path: str, query_image: str, topk: int = 5, device: str = "cpu", model_name: str = "resnet50") -> List[Tuple[str, float]]:
    """Performs the recommendation by searching for similar images in the index."""
    # Loads the existing index
    emb, paths = load_index(index_path)
    emb = normalize_rows(emb)
    
    # Generates the embedding of the query image
    embeder = ImageEmbedder(device=device, model_name=model_name)
    q = embeder.embed(query_image)
    q = q.astype(np.float32)
    
    # Normalizes the query vector
    qn = q / (np.linalg.norm(q) + 1e-12)
    
    # Computes similarity (dot product of normalized vectors = cosine similarity)
    scores = emb.dot(qn)
    
    # Sorts in descending order by the scores
    idx = np.argsort(-scores)[:max(1, topk)]
    
    # Returns the paths and scores of the top-k
    return [(paths[i], float(scores[i])) for i in idx]

### Data Output

In [104]:
def write_jsonl(items: List[Tuple[str, float]], out_path: str):
    """Writes the recommendation results in JSON Lines format."""
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for p, s in items:
            f.write(json.dumps({"path": p, "score": round(s, 6)}) + "\n")

### Results Visualization

In [105]:
def show_image(path: str, title: str = ""):
    """Displays an image using matplotlib."""
    img = Image.open(path).convert("RGB")
    plt.imshow(img)
    plt.title(title)
    plt.axis("off")

def plot_recommendations(query_path: str, recommendations: List[Tuple[str, float]]):
    """Plots the query image and the recommendations side by side."""
    n = len(recommendations)
    plt.figure(figsize=(15, 5))
    
    # Shows the query
    plt.subplot(1, n + 1, 1)
    show_image(query_path, "Query")
    
    # Shows the recommended items
    for i, (path, score) in enumerate(recommendations):
        plt.subplot(1, n + 1, i + 2)
        show_image(path, f"Similar: {score:.2f}")
    
    plt.tight_layout()
    plt.show()

### Command-Line Interface (CLI)

In [106]:
def make_parser() -> argparse.ArgumentParser:
    """Configures the command-line arguments."""
    parser = argparse.ArgumentParser(prog="image_recommender", description="Image recommendation system based on CNN embeddings")
    sub = parser.add_subparsers(dest="cmd", required=False)

    # Subcommand 'index'
    p_index = sub.add_parser("index", help="Index a directory of images")
    p_index.add_argument("--images", required=False, default=os.getenv("IMAGE_DIR", "images"))
    p_index.add_argument("--out", required=False, default=os.getenv("INDEX_PATH", "index.npz"))
    p_index.add_argument("--device", required=False, default=os.getenv("DEVICE", "cpu"))
    p_index.add_argument("--model", required=False, default=os.getenv("MODEL_NAME", "resnet50"))

    # Subcommand 'recommend'
    p_rec = sub.add_parser("recommend", help="Generate recommendations for a query image")
    p_rec.add_argument("--index", required=False, default=os.getenv("INDEX_PATH", "index.npz"))
    p_rec.add_argument("--query", required=True)
    p_rec.add_argument("--topk", type=int, required=False, default=5)
    p_rec.add_argument("--out", required=False, default="recommendations.jsonl")
    p_rec.add_argument("--device", required=False, default=os.getenv("DEVICE", "cpu"))
    p_rec.add_argument("--model", required=False, default=os.getenv("MODEL_NAME", "resnet50"))

    return parser

### Main Function

In [107]:
def main():
    """Orchestration function of the application."""
    parser = make_parser()
    args = parser.parse_args()
    
    # Checks whether a command was passed
    if not args.cmd:
        parser.print_help()
        sys.exit(0)
    
    # Runs the chosen command
    if args.cmd == "index":
        try:
            index_images(args.images, args.out, device=args.device, model_name=args.model)
        except Exception as e:
            print(json.dumps({"erro": str(e)}))
            sys.exit(1)
    elif args.cmd == "recommend":
        try:
            items = recommend(args.index, args.query, topk=int(args.topk), device=args.device, model_name=args.model)
            write_jsonl(items, args.out)
            print(json.dumps({"recomendacoes": len(items), "saida": args.out}))
        except Exception as e:
            print(json.dumps({"erro": str(e)}))
            sys.exit(1)
    else:
        parser.print_help()

## Interactive Demo

Use the interface below to pick an image from your dataset and see the visual recommendations in real time.

In [108]:
# Basic settings
images_dir = "./dataset"
index_path = "./dataset_index.npz"
device = os.getenv("cpu")
model_name = "resnet50"

# 1. Automatic indexing (if needed)
if not os.path.exists(index_path):
    print("Generating the initial index...")
    index_images(images_dir, index_path, device=device, model_name=model_name)

# 2. Interactive Interface
all_images = list_images(images_dir)
dropdown = widgets.Dropdown(
    options=all_images,
    description='Image:',
    disabled=False,
)

button = widgets.Button(description="Find Similar")
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        query = dropdown.value
        print(f"Processing: {query}...")
        try:
            results = recommend(index_path, query, topk=5, device=device, model_name=model_name)
            plot_recommendations(query, results)
        except Exception as e:
            print(f"Error: {e}")

button.on_click(on_button_clicked)

print("Select an image below and click the button:")
display(widgets.VBox([dropdown, button, output]))

Gerando índice inicial...
{"tempo_segundos": 4.229, "imagens_indexadas": 30, "dim": 2048}
Selecione uma imagem abaixo e clique no botão:
